# SAR Water Segmentation - Multi-Dataset Training
## Vision Transformer for Sentinel-1 Water Detection

This notebook supports:
- **Dataset 1**: Images (.tif) + Labels (.shp)
- **Dataset 2**: Sentinel-1 images + mask TIF files
- Automatic dataset detection and loading
- Combined training on both datasets

## 1. Setup & Installation

In [1]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

Mounted at /content/drive


In [2]:
# Install required packages
!pip install albumentations -q
!pip install geopandas -q
!pip install rasterio -q

In [3]:
# Imports
import os
import sys
import gc
import json
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, asdict
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from skimage.transform import resize

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Geospatial
import rasterio
from rasterio.features import rasterize
import geopandas as gpd

# Augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Configuration

In [61]:
# # @dataclass
# class Config:
#     """Training configuration"""

#     # ===== PATHS - UPDATE THESE =====
#     # Dataset 1: Shapefile-based
#     dataset1_dir: str = "/content/drive/MyDrive/SAR_water_images_and_labels"
#     dataset1_images: str = "images"  # subfolder name
#     dataset1_labels: str = "labels"   # subfolder name

#     # Dataset 2: Sentinel-1 with mask TIF files
#     dataset2_dir: str = "/content/drive/MyDrive/Sentinel1_Dataset"
#     # Will automatically scan for folders containing sentinel12_s1_*_img.tif and sentinel12_s1_*_msk.tif

#     # Output directory
#     output_dir: str = "/content/drive/MyDrive/SAR_water_combined_outputs"

#     # ===== DATASET SELECTION =====
#     use_dataset1: bool = False   # Use shapefile dataset
#     use_dataset2: bool = True   # Use Sentinel-1 dataset

#     # ===== MODEL & IMAGE PARAMETERS =====
#     img_size: int = 512
#     in_channels: int = 2  # Sentinel-1 dual-pol (VV, VH)
#     out_channels: int = 1

#     # ViT architecture
#     patch_size: int = 16
#     embed_dim: int = 256
#     depth: int = 6
#     num_heads: int = 8
#     mlp_ratio: float = 4.0

#     # ===== TRAINING PARAMETERS =====
#     batch_size: int = 2
#     accumulate_steps: int = 4  # Effective batch = 8
#     num_epochs: int = 70
#     learning_rate: float = 1e-2
#     weight_decay: float = 1e-5
#     warmup_epochs: int = 5

#     # Optimization
#     lr_patience: int = 5
#     lr_factor: float = 0.5
#     lr_min: float = 1e-7
#     early_stop_patience: int = 15

#     # Training settings
#     use_amp: bool = True
#     augment: bool = True
#     num_workers: int = 0 # Changed from 2 to 0 to prevent DataLoader memory issues in Colab

#     # Data split
#     val_split: float = 0.2  # 80/20 train/val split
#     seed: int = 42

#     def __post_init__(self):
#         """Create output directories"""
#         for subdir in ["", "checkpoints", "metrics", "predictions"]:
#             os.makedirs(os.path.join(self.output_dir, subdir), exist_ok=True)

# # Initialize config
# config = Config()
# print("✅ Configuration loaded")
# print(f"   Using Dataset 1 (Shapefile): {config.use_dataset1}")
# print(f"   Using Dataset 2 (Sentinel-1): {config.use_dataset2}")


@dataclass
class Config:
    """Training configuration"""

    # ===== PATHS =====
    dataset1_dir: str = "/content/drive/MyDrive/SAR_water_images_and_labels"
    dataset1_images: str = "images"
    dataset1_labels: str = "labels"
    dataset2_dir: str = "/content/drive/MyDrive/Sentinel1_Dataset"
    output_dir: str = "/content/drive/MyDrive/SAR_water_combined_outputs"

    # ===== DATASET SELECTION =====
    use_dataset1: bool = False   # Changed to False (no shapefile data)
    use_dataset2: bool = True    # Use Sentinel-1 data

    # ===== PATCH-BASED TRAINING SETTINGS =====
    img_size: int = 512  # Patch size
    patches_per_image: int = 20  # Random patches per large image
    val_patches_per_image: int = 100
    in_channels: int = 2
    out_channels: int = 1

    # ===== MODEL ARCHITECTURE =====
    patch_size: int = 16
    embed_dim: int = 256
    depth: int = 6
    num_heads: int = 8
    mlp_ratio: float = 4.0

    # ===== TRAINING PARAMETERS =====
    batch_size: int = 8  # Larger batch for patches
    accumulate_steps: int = 2  # Effective batch = 16
    num_epochs: int = 70
    learning_rate: float = 5e-5  # Lower LR for patches
    weight_decay: float = 1e-2
    warmup_epochs: int = 5

    # Optimization
    lr_patience: int = 5
    lr_factor: float = 0.5
    lr_min: float = 1e-7
    early_stop_patience: int = 15

    # Training settings
    use_amp: bool = True
    augment: bool = True
    # num_workers: int = 2
    # pin_memory: bool = True  # ✅ This is present
    num_workers: int = 0
    pin_memory: bool = False   # May try these

    # Data split
    val_split: float = 0.2
    seed: int = 42

    def __post_init__(self):
        """Create output directories"""
        for subdir in ["", "checkpoints", "metrics", "predictions"]:
            os.makedirs(os.path.join(self.output_dir, subdir), exist_ok=True)

# Initialize config
config = Config()
print("✅ Configuration loaded")
print(f"   Using Dataset 1 (Shapefile): {config.use_dataset1}")
print(f"   Using Dataset 2 (Sentinel-1): {config.use_dataset2}")
print(f"   Patches per image: {config.patches_per_image}")
print(f"   Batch size: {config.batch_size}")
print(f"   Effective batch size: {config.batch_size * config.accumulate_steps}")

✅ Configuration loaded
   Using Dataset 1 (Shapefile): False
   Using Dataset 2 (Sentinel-1): True
   Patches per image: 20
   Batch size: 8
   Effective batch size: 16


## 3. Utility Functions

In [62]:
def set_seed(seed: int = 42):
    """Set random seeds for reproducibility"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def get_device() -> torch.device:
    """Get available compute device"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"✅ Using device: {device}")
    return device

set_seed(config.seed)
DEVICE = get_device()

✅ Using device: cuda


## 4. Data Augmentation

In [63]:
def get_training_augmentation() -> A.Compose:
    """SAR-optimized training augmentation - geometric only"""
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(
            translate_percent={"x": (-0.1, 0.1), "y": (-0.1, 0.1)},
            scale={"x": (0.9, 1.1), "y": (0.9, 1.1)},
            rotate=(-15, 15),
            p=0.5
        ),
        ToTensorV2()
    ])

def get_validation_augmentation() -> A.Compose:
    """Validation augmentation - geometric only (tensor conversion only)"""
    return A.Compose([
        ToTensorV2()
    ])

## 5. Dataset Classes

In [64]:
class ShapefileDataset(Dataset):
    """
    Dataset 1: Images with shapefile labels
    Converts shapefiles to raster masks on-the-fly
    """

    def __init__(self, image_dir, mask_dir, file_list, transform, in_channels=2, img_size=512):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.files = file_list
        self.transform = transform
        self.in_channels = in_channels
        self.img_size = img_size

        # Index available shapefiles
        if os.path.exists(mask_dir):
            self.available_shapes = {
                os.path.splitext(f)[0]: f
                for f in os.listdir(mask_dir)
                if f.lower().endswith('.shp')
            }
        else:
            self.available_shapes = {}

    def __len__(self):
        return len(self.files)

    def _rasterize_shapefile(self, shp_path, transform, crs, height, width):
        """Convert shapefile to raster mask"""
        try:
            gdf = gpd.read_file(shp_path)
            if gdf.crs != crs:
                gdf = gdf.to_crs(crs)

            mask = rasterize(
                shapes=[(geom, 1) for geom in gdf.geometry if geom is not None],
                out_shape=(height, width),
                transform=transform,
                fill=0,
                dtype='uint8'
            )
            return mask.astype(np.float32)
        except Exception as e:
            print(f"⚠️  Failed to rasterize {shp_path}: {e}")
            return np.zeros((height, width), dtype=np.float32)

    def __getitem__(self, idx):
        img_name = self.files[idx]
        img_path = os.path.join(self.image_dir, img_name)

        # Load SAR image
        with rasterio.open(img_path) as src:
            bands = list(range(1, min(self.in_channels + 1, src.count + 1)))
            image = src.read(bands).transpose(1, 2, 0).astype(np.float32)

            # Ensure we have exactly in_channels
            if image.shape[2] < self.in_channels:
                # Pad with zeros if fewer channels
                pad = np.zeros((image.shape[0], image.shape[1], self.in_channels - image.shape[2]), dtype=np.float32)
                image = np.concatenate([image, pad], axis=2)
            elif image.shape[2] > self.in_channels:
                # Take first in_channels if more
                image = image[:, :, :self.in_channels]

            if image.max() > 1.0:
                image = image / 255.0

            ref_transform = src.transform
            ref_crs = src.crs
            ref_height = src.height
            ref_width = src.width

        # Find and rasterize shapefile
        base_name = os.path.splitext(img_name)[0]
        shp_filename = self.available_shapes.get(base_name)

        if shp_filename:
            shp_path = os.path.join(self.mask_dir, shp_filename)
            mask = self._rasterize_shapefile(shp_path, ref_transform, ref_crs, ref_height, ref_width)
        else:
            mask = np.zeros((ref_height, ref_width), dtype=np.float32)

        # Resize if needed
        if image.shape[0] != self.img_size or image.shape[1] != self.img_size:
            image = resize(image, (self.img_size, self.img_size, self.in_channels),
                         mode='reflect', anti_aliasing=True, preserve_range=True).astype(np.float32)
            mask = resize(mask, (self.img_size, self.img_size),
                        order=0, mode='reflect', anti_aliasing=False, preserve_range=True).astype(np.float32)
            mask = (mask > 0.5).astype(np.float32)

        # Apply transforms
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image, mask = transformed['image'], transformed['mask']

        if mask.dim() == 2:
            mask = mask.unsqueeze(0)

        return image, mask


# class Sentinel1Dataset(Dataset):
#     """
#     Dataset 2: Sentinel-1 images with mask TIF files
#     Format: sentinel12_s1_XX_img.tif and sentinel12_s1_XX_msk.tif
#     """

#     def __init__(self, data_pairs, transform, in_channels=2, img_size=512):
#         """
#         Args:
#             data_pairs: List of tuples [(img_path, mask_path), ...]
#             transform: Albumentations transform
#             in_channels: Number of input channels (2 for SAR)
#             img_size: Target image size
#         """
#         self.data_pairs = data_pairs
#         self.transform = transform
#         self.in_channels = in_channels
#         self.img_size = img_size

#     def __len__(self):
#         return len(self.data_pairs)

#     def __getitem__(self, idx):
#         img_path, mask_path = self.data_pairs[idx]

#         # Load Sentinel-1 image
#         with rasterio.open(img_path) as src:
#             # Read first in_channels bands
#             bands = list(range(1, min(self.in_channels + 1, src.count + 1)))
#             image = src.read(bands).transpose(1, 2, 0).astype(np.float32)

#             # Ensure we have exactly in_channels
#             if image.shape[2] < self.in_channels:
#                 pad = np.zeros((image.shape[0], image.shape[1], self.in_channels - image.shape[2]), dtype=np.float32)
#                 image = np.concatenate([image, pad], axis=2)
#             elif image.shape[2] > self.in_channels:
#                 image = image[:, :, :self.in_channels]

#             # Normalize if needed
#             if image.max() > 1.0:
#                 image = image / 255.0

#         # Load mask
#         with rasterio.open(mask_path) as src:
#             mask = src.read(1).astype(np.float32)
#             # Binarize mask (assuming water=1, non-water=0)
#             mask = (mask > 0).astype(np.float32)

#         # Resize if needed
#         if image.shape[0] != self.img_size or image.shape[1] != self.img_size:
#             image = resize(image, (self.img_size, self.img_size, self.in_channels),
#                          mode='reflect', anti_aliasing=True, preserve_range=True).astype(np.float32)
#             mask = resize(mask, (self.img_size, self.img_size),
#                         order=0, mode='reflect', anti_aliasing=False, preserve_range=True).astype(np.float32)
#             mask = (mask > 0.5).astype(np.float32)

#         # Apply transforms
#         if self.transform:
#             transformed = self.transform(image=image, mask=mask)
#             image, mask = transformed['image'], transformed['mask']

#         if mask.dim() == 2:
#             mask = mask.unsqueeze(0)

#         return image, mask


class LargeImageDataset(Dataset):
    """Dataset for large images - extracts random patches on-the-fly"""

    def __init__(self, data_pairs, transform, patch_size=512,
                 in_channels=2, num_patches_per_image=10):
        self.data_pairs = data_pairs
        self.transform = transform
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.num_patches_per_image = num_patches_per_image  # Patches to extract per image

        # Pre-load image bounds without loading full images
        self.image_info = []
        for img_path, mask_path in data_pairs:
            with rasterio.open(img_path) as src:
                self.image_info.append({
                    'img_path': img_path,
                    'mask_path': mask_path,
                    'height': src.height,
                    'width': src.width,
                    'transform': src.transform,
                    'crs': src.crs
                })

    def __len__(self):
        return len(self.data_pairs) * self.num_patches_per_image

    def __getitem__(self, idx):
        # Which image and which patch
        img_idx = idx // self.num_patches_per_image
        patch_idx = idx % self.num_patches_per_image

        info = self.image_info[img_idx]

        # Random patch location
        max_y = info['height'] - self.patch_size
        max_x = info['width'] - self.patch_size

        if max_y > 0 and max_x > 0:
            y = np.random.randint(0, max_y)
            x = np.random.randint(0, max_x)
        else:
            y, x = 0, 0

        # Read only the patch window (memory efficient!)
        window = rasterio.windows.Window(x, y, self.patch_size, self.patch_size)

        # Load image patch
        with rasterio.open(info['img_path']) as src:
            bands = list(range(1, min(self.in_channels + 1, src.count + 1)))
            image = src.read(bands, window=window).transpose(1, 2, 0).astype(np.float32)

        # Load mask patch
        with rasterio.open(info['mask_path']) as src:
            mask = src.read(1, window=window).astype(np.float32)
            mask = (mask > 0).astype(np.float32)

        # Normalize
        if image.max() > 1.0:
            image = image / 255.0

        # Apply transforms
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image, mask = transformed['image'], transformed['mask']

        if mask.dim() == 2:
            mask = mask.unsqueeze(0)

        return image, mask


print("✅ Dataset classes defined")

✅ Dataset classes defined


## 6. Dataset Discovery & Loading

In [65]:
def find_sentinel1_data(root_dir):
    """
    Recursively find Sentinel-1 image and mask pairs
    Looks for: sentinel12_s1_*_img.tif and sentinel12_s1_*_msk.tif
    """
    data_pairs = []

    for root, dirs, files in os.walk(root_dir):
        # Find all S1 image files
        img_files = [f for f in files if '_s1_' in f and '_img.tif' in f]

        for img_file in img_files:
            # Construct corresponding mask filename
            # Example: sentinel12_s1_80_img.tif -> sentinel12_s1_80_msk.tif
            mask_file = img_file.replace('_img.tif', '_msk.tif')

            img_path = os.path.join(root, img_file)
            mask_path = os.path.join(root, mask_file)

            if os.path.exists(mask_path):
                data_pairs.append((img_path, mask_path))
            else:
                print(f"⚠️  Mask not found for {img_file}")

    return data_pairs


def load_datasets(config):
    """
    Load and combine datasets based on configuration
    """
    train_datasets = []
    val_datasets = []

    # ===== Dataset 1: Shapefile-based =====
    if config.use_dataset1:
        print("\n📁 Loading Dataset 1 (Shapefile-based)...")

        img_dir = os.path.join(config.dataset1_dir, config.dataset1_images)
        mask_dir = os.path.join(config.dataset1_dir, config.dataset1_labels)

        if os.path.exists(img_dir):
            all_files = sorted([
                f for f in os.listdir(img_dir)
                if f.lower().endswith(('.tif', '.tiff', '.png', '.jpg', '.jpeg'))
            ])

            if all_files:
                # Split train/val
                rng = np.random.RandomState(config.seed)
                rng.shuffle(all_files)
                split_idx = int((1 - config.val_split) * len(all_files))
                train_files = all_files[:split_idx]
                val_files = all_files[split_idx:]

                # Create datasets
                train_ds1 = ShapefileDataset(
                    img_dir, mask_dir, train_files,
                    get_training_augmentation() if config.augment else get_validation_augmentation(),
                    config.in_channels, config.img_size
                )
                val_ds1 = ShapefileDataset(
                    img_dir, mask_dir, val_files,
                    get_validation_augmentation(),
                    config.in_channels, config.img_size
                )

                train_datasets.append(train_ds1)
                val_datasets.append(val_ds1)

                print(f"   ✅ Found {len(all_files)} images")
                print(f"   ✅ Split: {len(train_files)} train, {len(val_files)} val")
            else:
                print(f"   ⚠️  No images found in {img_dir}")
        else:
            print(f"   ⚠️  Directory not found: {img_dir}")

    # ===== Dataset 2: Sentinel-1 with mask TIF =====
    # if config.use_dataset2:
    #     print("\n📁 Loading Dataset 2 (Sentinel-1)...")

    #     if os.path.exists(config.dataset2_dir):
    #         # Find all Sentinel-1 data pairs
    #         data_pairs = find_sentinel1_data(config.dataset2_dir)

    #         if data_pairs:
    #             # Split train/val
    #             rng = np.random.RandomState(config.seed)
    #             rng.shuffle(data_pairs)
    #             split_idx = int((1 - config.val_split) * len(data_pairs))
    #             train_pairs = data_pairs[:split_idx]
    #             val_pairs = data_pairs[split_idx:]

    #             # Create datasets
    #             train_ds2 = Sentinel1Dataset(
    #                 train_pairs,
    #                 get_training_augmentation() if config.augment else get_validation_augmentation(),
    #                 config.in_channels, config.img_size
    #             )
    #             val_ds2 = Sentinel1Dataset(
    #                 val_pairs,
    #                 get_validation_augmentation(),
    #                 config.in_channels, config.img_size
    #             )

    #             train_datasets.append(train_ds2)
    #             val_datasets.append(val_ds2)

    #             print(f"   ✅ Found {len(data_pairs)} Sentinel-1 pairs")
    #             print(f"   ✅ Split: {len(train_pairs)} train, {len(val_pairs)} val")
    #         else:
    #             print(f"   ⚠️  No Sentinel-1 data found in {config.dataset2_dir}")
    #     else:
    #         print(f"   ⚠️  Directory not found: {config.dataset2_dir}")

    # ===== Dataset 2: Large Sentinel-1 with patch extraction =====
    if config.use_dataset2:
        print("\n📁 Loading Dataset 2 (Large Sentinel-1 with patch extraction)...")

        if os.path.exists(config.dataset2_dir):
            # Find all Sentinel-1 data pairs
            data_pairs = find_sentinel1_data(config.dataset2_dir)

            if data_pairs:
                # Split train/val
                rng = np.random.RandomState(config.seed)
                rng.shuffle(data_pairs)
                split_idx = int((1 - config.val_split) * len(data_pairs))
                train_pairs = data_pairs[:split_idx]
                val_pairs = data_pairs[split_idx:]

                print(f"   Found {len(data_pairs)} large images")
                print(f"   Train images: {len(train_pairs)}")
                print(f"   Val images: {len(val_pairs)}")
                print(f"   Patches per image: {config.patches_per_image}")
                print(f"   Total train patches: {len(train_pairs) * config.patches_per_image}")
                print(f"   Total val patches: {len(val_pairs) * config.patches_per_image}")

                # Use LargeImageDataset instead of Sentinel1Dataset
                train_ds2 = LargeImageDataset(
                    train_pairs,
                    get_training_augmentation() if config.augment else get_validation_augmentation(),
                    patch_size=config.img_size,
                    in_channels=config.in_channels,
                    num_patches_per_image=config.patches_per_image
                )

                # Then in load_datasets:
                val_ds2 = LargeImageDataset(
                    val_pairs,
                    get_validation_augmentation(),
                    patch_size=config.img_size,
                    in_channels=config.in_channels,
                    num_patches_per_image=config.val_patches_per_image  # 100 patches per image
                )

                train_datasets.append(train_ds2)
                val_datasets.append(val_ds2)

                print(f"   ✅ Loaded {len(train_ds2)} training patches")
                print(f"   ✅ Loaded {len(val_ds2)} validation patches")
            else:
                print(f"   ⚠️  No Sentinel-1 data found in {config.dataset2_dir}")
        else:
            print(f"   ⚠️  Directory not found: {config.dataset2_dir}")

    # Combine datasets
    if not train_datasets:
        raise ValueError("No datasets loaded! Check your configuration and paths.")

    train_dataset = ConcatDataset(train_datasets) if len(train_datasets) > 1 else train_datasets[0]
    val_dataset = ConcatDataset(val_datasets) if len(val_datasets) > 1 else val_datasets[0]

    print(f"\n✅ Combined Dataset:")
    print(f"   Total Training: {len(train_dataset)} samples")
    print(f"   Total Validation: {len(val_dataset)} samples")

    return train_dataset, val_dataset



    # Combine datasets
    # if not train_datasets:
    #     raise ValueError("No datasets loaded! Check your configuration and paths.")

    # train_dataset = ConcatDataset(train_datasets) if len(train_datasets) > 1 else train_datasets[0]
    # val_dataset = ConcatDataset(val_datasets) if len(val_datasets) > 1 else val_datasets[0]

    # print(f"\n✅ Combined Dataset:")
    # print(f"   Total Training: {len(train_dataset)} samples")
    # print(f"   Total Validation: {len(val_dataset)} samples")

    # return train_dataset, val_dataset


# Load datasets
train_dataset, val_dataset = load_datasets(config)

# # Create dataloaders
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=config.batch_size,
#     shuffle=True,
#     num_workers=config.num_workers,
#     pin_memory=True,
#     drop_last=True
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=config.batch_size,
#     shuffle=False,
#     num_workers=config.num_workers,
#     pin_memory=True
# )

# print(f"\n✅ DataLoaders created")
# print(f"   Train batches: {len(train_loader)}")
# print(f"   Val batches: {len(val_loader)}")



# Create dataloaders (Section 6)
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory,
    drop_last=True  # Keep drop_last=True for consistent batch sizes
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory
)

print(f"\n✅ DataLoaders created")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")


📁 Loading Dataset 2 (Large Sentinel-1 with patch extraction)...
   Found 7 large images
   Train images: 5
   Val images: 2
   Patches per image: 20
   Total train patches: 100
   Total val patches: 40
   ✅ Loaded 100 training patches
   ✅ Loaded 200 validation patches

✅ Combined Dataset:
   Total Training: 100 samples
   Total Validation: 200 samples

✅ DataLoaders created
   Train batches: 12
   Val batches: 25


In [66]:
# List contents of your Drive to find the correct paths
base_path = "/content/drive/MyDrive/"
print("Contents of MyDrive:")
for item in os.listdir(base_path):
    item_path = os.path.join(base_path, item)
    if os.path.isdir(item_path):
        print(f"  📁 {item}")
    else:
        print(f"  📄 {item}")

Contents of MyDrive:
  📁 Colab Notebooks
  📁 ESOL and IELTS materials
  📄 Bayram_Bayramov_wvariant_B.docx
  📄 Bayram_Bayramov_evariant_B.xlsx
  📄 WhatsApp Image 2021-06-29 at 6.51.38 AM.jpeg
  📄 peyvend-sertifikati-sars-cov-2 (1).pdf
  📄 peyvend-sertifikati-sars-cov-2 (1) (1).pdf
  📄 IMG_20221008_202347.jpg
  📄 Bayram's Resume.pdf
  📄 internprez_.drawio
  📄 Screenshot_2024-10-16-13-16-48-552_iba.mobilbank.jpg
  📄 Template 1.pdf
  📄 Conglomerate Concrete Crack Detection.zip
  📁 .ipynb_checkpoints
  📄 Bayram_Bayramov_Resume.pdf
  📄 python questions.gdoc
  📄 Bayram Bayramov.gdoc
  📄 Новый документ (1).gdoc
  📄 Новая таблица.gsheet
  📄 LeetCode questions.gdoc
  📄 Новый документ.gdoc
  📄 questions.gdoc
  📄 VID_20250812_173542.mp4
  📄 VID_20250812_173306.mp4
  📄 VID_20250812_173148.mp4
  📄 VID_20250813_125752.mp4
  📄 VID_20250813_125626.mp4
  📄 Statistical Modeling and Function Approximation.gsheet
  📁 Google Earth
  📁 SAR_water_images_and_labels
  📁 Sentinel1_Dataset
  📁 SAR_water_combine

## 7. Visualize Sample Data

In [67]:
# from google.colab import drive
# drive.mount('/content/drive')

In [68]:
# # Visualize a few samples
# def visualize_samples(loader, num_samples=2):
#     """Visualize sample images and masks"""
#     images, masks = next(iter(loader))

#     fig, axes = plt.subplots(num_samples, 3, figsize=(12, num_samples * 3))
#     if num_samples == 1:
#         axes = axes.reshape(1, -1)

#     for i in range(min(num_samples, len(images))):
#         img = images[i].cpu().numpy().transpose(1, 2, 0)
#         mask = masks[i].cpu().numpy().squeeze()

#         # Denormalize
#         img = (img * 0.5 + 0.5).clip(0, 1)

#         # Display VV channel (first band)
#         axes[i, 0].imshow(img[:, :, 0], cmap='gray')
#         axes[i, 0].set_title(f'Sample {i+1} - VV')
#         axes[i, 0].axis('off')

#         # Display VH channel (second band)
#         axes[i, 1].imshow(img[:, :, 1], cmap='gray')
#         axes[i, 1].set_title(f'Sample {i+1} - VH')
#         axes[i, 1].axis('off')

#         # Display mask
#         axes[i, 2].imshow(mask, cmap='Blues')
#         axes[i, 2].set_title(f'Sample {i+1} - Mask')
#         axes[i, 2].axis('off')

#     plt.tight_layout()
#     plt.show()

# visualize_samples(train_loader, num_samples=4)

## 8. Model Architecture

In [69]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=512, patch_size=16, in_channels=2, embed_dim=256):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x


class MultiHeadAttention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=8, mlp_ratio=4., qkv_bias=True, drop=0., attn_drop=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadAttention(dim, num_heads, qkv_bias, attn_drop, drop)
        self.norm2 = nn.LayerNorm(dim)

        mlp_hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(mlp_hidden, dim),
            nn.Dropout(drop)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class ViTEncoder(nn.Module):
    def __init__(self, img_size=512, patch_size=16, in_channels=2, embed_dim=256, depth=6, num_heads=8):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)

        num_patches = (img_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio=4.0)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        nn.init.normal_(self.pos_embed, mean=0.0, std=0.02)

    def forward(self, x):
        x = self.patch_embed(x) + self.pos_embed
        for block in self.blocks:
            x = block(x)
        return self.norm(x)


class ViTSegmentation(nn.Module):
    def __init__(self, img_size=512, patch_size=16, in_channels=2, num_classes=1,
                 embed_dim=256, depth=6, num_heads=8):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size

        self.encoder = ViTEncoder(img_size, patch_size, in_channels, embed_dim, depth, num_heads)

        self.head = nn.Sequential(
            nn.Conv2d(embed_dim, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64, num_classes, kernel_size=1)
        )

    def forward(self, x):
        B = x.shape[0]
        x = self.encoder(x)

        H = W = self.img_size // self.patch_size
        x = x.transpose(1, 2).reshape(B, -1, H, W)

        x = self.head(x)
        x = F.interpolate(x, size=(self.img_size, self.img_size),
                         mode='bilinear', align_corners=True)
        return x


print("✅ Model architecture defined")

✅ Model architecture defined


## 9. Loss Functions & Metrics

In [70]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = torch.sigmoid(preds).view(-1)
        targets = targets.view(-1)
        intersection = (preds * targets).sum()
        dice = (2. * intersection + self.smooth) / (preds.sum() + targets.sum() + self.smooth)
        return 1 - dice


class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, preds, targets):
        return self.bce_weight * self.bce(preds, targets) + self.dice_weight * self.dice(preds, targets)


def calculate_metrics(preds, targets, threshold=0.5):
    """Calculate segmentation metrics"""
    if preds.min() < 0 or preds.max() > 1:
        preds = torch.sigmoid(preds)

    preds_bin = (preds > threshold).float().view(-1)
    targets = targets.view(-1)

    tp = (preds_bin * targets).sum()
    fp = (preds_bin * (1 - targets)).sum()
    fn = ((1 - preds_bin) * targets).sum()
    tn = ((1 - preds_bin) * (1 - targets)).sum()

    eps = 1e-7

    return {
        'iou': (tp + eps) / (tp + fp + fn + eps),
        'dice': (2 * tp + eps) / (2 * tp + fp + fn + eps),
        'precision': (tp + eps) / (tp + fp + eps),
        'recall': (tp + eps) / (tp + fn + eps),
        'accuracy': (tp + tn + eps) / (tp + tn + fp + fn + eps),
        'f1_score': (2 * (tp + eps) / (tp + fp + eps) * (tp + eps) / (tp + fn + eps)) /
                   ((tp + eps) / (tp + fp + eps) + (tp + eps) / (tp + fn + eps) + eps)
    }


print("✅ Loss functions & metrics defined")

✅ Loss functions & metrics defined


## 10. Training Functions

In [71]:
def train_one_epoch(model, loader, criterion, optimizer, device, scaler, accumulate_steps):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    metrics_sum = {k: 0.0 for k in ['iou', 'dice', 'precision', 'recall', 'accuracy', 'f1_score']}

    optimizer.zero_grad()
    pbar = tqdm(loader, desc="Training")

    for i, (images, masks) in enumerate(pbar):
        images, masks = images.to(device), masks.to(device)
        loss_scale = 1.0 / accumulate_steps

        if scaler:
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, masks) * loss_scale
            scaler.scale(loss).backward()
        else:
            outputs = model(images)
            loss = criterion(outputs, masks) * loss_scale
            loss.backward()

        if (i + 1) % accumulate_steps == 0 or (i + 1) == len(loader):
            if scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()

        with torch.no_grad():
            batch_metrics = calculate_metrics(outputs.detach(), masks)
            for k in metrics_sum:
                metrics_sum[k] += batch_metrics[k]

        running_loss += loss.item() * accumulate_steps
        pbar.set_postfix(loss=loss.item() * accumulate_steps, dice=batch_metrics['dice'])

    avg_loss = running_loss / len(loader)
    avg_metrics = {k: v / len(loader) for k, v in metrics_sum.items()}

    return avg_loss, avg_metrics


def validate(model, loader, criterion, device):
    """Validate model"""
    model.eval()
    running_loss = 0.0
    metrics_sum = {k: 0.0 for k in ['iou', 'dice', 'precision', 'recall', 'accuracy', 'f1_score']}

    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Validation"):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)

            batch_metrics = calculate_metrics(outputs, masks)
            for k in metrics_sum:
                metrics_sum[k] += batch_metrics[k]

            running_loss += loss.item()

    avg_loss = running_loss / len(loader)
    avg_metrics = {k: v / len(loader) for k, v in metrics_sum.items()}

    return avg_loss, avg_metrics


def lr_warmup(epoch, base_lr, warmup_epochs):
    """Linear learning rate warmup"""
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs
    return base_lr


print("✅ Training functions defined")

✅ Training functions defined


## 11. Initialize Model & Optimizer

In [72]:
# Initialize model
model = ViTSegmentation(
    config.img_size,
    config.patch_size,
    config.in_channels,
    config.out_channels,
    config.embed_dim,
    config.depth,
    config.num_heads
).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model initialized")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

# Loss, optimizer, scheduler
criterion = BCEDiceLoss()
optimizer = Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
scheduler = ReduceLROnPlateau(
    optimizer, mode='min', factor=config.lr_factor,
    patience=config.lr_patience, min_lr=config.lr_min
)

scaler = torch.cuda.amp.GradScaler() if config.use_amp and torch.cuda.is_available() else None

print("✅ Optimizer & scheduler ready")

✅ Model initialized
   Total parameters: 5,501,825
   Trainable parameters: 5,501,825
✅ Optimizer & scheduler ready


## 12. Training Loop

In [73]:
# Check what files were found
print("\n=== DATASET DEBUGGING ===")

# Check Dataset 1
import os
from pathlib import Path

if config.use_dataset1:
    img_dir = os.path.join(config.dataset1_dir, config.dataset1_images)
    mask_dir = os.path.join(config.dataset1_dir, config.dataset1_labels)

    print(f"\nDataset 1 - Shapefile:")
    print(f"  Image dir: {img_dir}")
    print(f"  Exists: {os.path.exists(img_dir)}")

    if os.path.exists(img_dir):
        images = [f for f in os.listdir(img_dir) if f.lower().endswith(('.tif', '.tiff', '.png'))]
        print(f"  Images found: {len(images)}")
        if len(images) > 0:
            print(f"  First 3: {images[:3]}")

    if os.path.exists(mask_dir):
        masks = [f for f in os.listdir(mask_dir) if f.endswith('.shp')]
        print(f"  Shapefiles found: {len(masks)}")

# Check Dataset 2
if config.use_dataset2:
    print(f"\nDataset 2 - Sentinel-1:")
    print(f"  Root dir: {config.dataset2_dir}")
    print(f"  Exists: {os.path.exists(config.dataset2_dir)}")

    if os.path.exists(config.dataset2_dir):
        # Count pairs
        pairs = find_sentinel1_data(config.dataset2_dir)
        print(f"  Image-mask pairs found: {len(pairs)}")
        if len(pairs) > 0:
            print(f"  First pair: {pairs[0]}")


# Print actual dataset sizes
print(f"\n=== DATASET SIZES ===")
print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Check if datasets are empty
if len(train_dataset) == 0:
    print("\n❌ ERROR: Training dataset is EMPTY!")
    print("Check your dataset paths and file formats.")







=== DATASET DEBUGGING ===

Dataset 2 - Sentinel-1:
  Root dir: /content/drive/MyDrive/Sentinel1_Dataset
  Exists: True
  Image-mask pairs found: 7
  First pair: ('/content/drive/MyDrive/Sentinel1_Dataset/S1_S2_Data/76/sentinel12_s1_76_img.tif', '/content/drive/MyDrive/Sentinel1_Dataset/S1_S2_Data/76/sentinel12_s1_76_msk.tif')

=== DATASET SIZES ===
Train dataset: 100 samples
Val dataset: 200 samples
Train batches: 12
Val batches: 25


In [ ]:
# import sys
# sys.exit()
# Training history
history = {
    'train_loss': [], 'val_loss': [],
    'train_dice': [], 'val_dice': [],
    'train_iou': [], 'val_iou': [],
    'lrs': []
}

best_val_loss = float('inf')
best_val_dice = 0.0
no_improvement = 0
best_checkpoint_path = os.path.join(config.output_dir, "checkpoints", "best_model.pth")

print("\n" + "="*70)
print("🚀 Starting Training")
print("="*70)
print(f"Epochs: {config.num_epochs}")
print(f"Batch Size: {config.batch_size} (effective: {config.batch_size * config.accumulate_steps})")
print(f"Learning Rate: {config.learning_rate}")
print(f"Image Size: {config.img_size}x{config.img_size}")
print(f"Channels: {config.in_channels}")
print("="*70 + "\n")

for epoch in range(config.num_epochs):
    # Learning rate warmup
    current_lr = lr_warmup(epoch, config.learning_rate, config.warmup_epochs)
    for param_group in optimizer.param_groups:
        param_group['lr'] = current_lr
    history['lrs'].append(current_lr)

    # Train
    train_loss, train_metrics = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE, scaler, config.accumulate_steps
    )

    # Validate
    val_loss, val_metrics = validate(model, val_loader, criterion, DEVICE)

    # Update scheduler
    if epoch >= config.warmup_epochs:
        scheduler.step(val_loss)

    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_dice'].append(train_metrics['dice'])
    history['val_dice'].append(val_metrics['dice'])
    history['train_iou'].append(train_metrics['iou'])
    history['val_iou'].append(val_metrics['iou'])

    # Print epoch summary
    print(
        f"Epoch {epoch+1}/{config.num_epochs} | "
        f"LR: {current_lr:.2e} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Train Dice: {train_metrics['dice']:.4f} | Val Dice: {val_metrics['dice']:.4f} | "
        f"Val IoU: {val_metrics['iou']:.4f}"
    )

    # Save best model
    if val_metrics['dice'] > best_val_dice:
        best_val_loss = val_loss
        best_val_dice = val_metrics['dice']
        no_improvement = 0

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_dice': val_metrics['dice'],
            'config': asdict(config)
        }, best_checkpoint_path)

        print(f"  💾 Best model saved! Val Dice: {val_metrics['dice']:.4f}")
    else:
        no_improvement += 1

        if no_improvement >= config.early_stop_patience:
            print(f"\n🛑 Early stopping at epoch {epoch+1}")
            break

# Clean up
torch.cuda.empty_cache()
gc.collect()

print("\n" + "="*70)
print("✅ Training Complete!")
print(f"Best Validation Loss: {best_val_loss:.4f}")
print(f"Best Validation Dice: {best_val_dice:.4f}")
print("="*70)


🚀 Starting Training
Epochs: 70
Batch Size: 8 (effective: 16)
Learning Rate: 5e-05
Image Size: 512x512
Channels: 2



Training:   0%|          | 0/12 [00:00<?, ?it/s]

Validation:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 1/70 | LR: 1.00e-05 | Train Loss: 0.7085 | Val Loss: 0.7847 | Train Dice: 0.2206 | Val Dice: 0.0021 | Val IoU: 0.0011
  💾 Best model saved! Val Dice: 0.0021


Training:   0%|          | 0/12 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 4/70 | LR: 4.00e-05 | Train Loss: 0.5150 | Val Loss: 0.7317 | Train Dice: 0.7618 | Val Dice: 0.1098 | Val IoU: 0.0753
  💾 Best model saved! Val Dice: 0.1098


Training:   0%|          | 0/12 [00:00<?, ?it/s]

Validation:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 5/70 | LR: 5.00e-05 | Train Loss: 0.5676 | Val Loss: 0.8301 | Train Dice: 0.5650 | Val Dice: 0.1154 | Val IoU: 0.0725


In [ ]:
print("=== CHECKING VALIDATION IMAGES ===\n")

# Get validation pairs
data_pairs = find_sentinel1_data(config.dataset2_dir)
split_idx = int((1 - config.val_split) * len(data_pairs))
val_pairs = data_pairs[split_idx:]

for img_path, mask_path in val_pairs:
    print(f"\nImage: {os.path.basename(img_path)}")

    with rasterio.open(mask_path) as src:
        mask_full = src.read(1)
        if mask_full.max() > 1:
            mask_full = mask_full / 255.0

        water_ratio = (mask_full > 0.5).mean()
        print(f"  Full image water percentage: {water_ratio:.2%}")

        # Check random patches
        h, w = mask_full.shape
        patch_size = 512
        water_patches = 0

        for i in range(20):
            y = np.random.randint(0, max(1, h - patch_size))
            x = np.random.randint(0, max(1, w - patch_size))
            patch = mask_full[y:y+patch_size, x:x+patch_size]
            if patch.mean() > 0.01:  # At least 1% water
                water_patches += 1

        print(f"  Patches with water (>1%): {water_patches}/20 ({water_patches*5:.0f}%)")

## 13. Visualize Training History

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0, 0].set_title('Train vs Val Loss', fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Dice
axes[0, 1].plot(history['train_dice'], label='Train', linewidth=2)
axes[0, 1].plot(history['val_dice'], label='Val', linewidth=2)
axes[0, 1].set_title('Train vs Val Dice', fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Dice Score')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# IoU
axes[1, 0].plot(history['train_iou'], label='Train', linewidth=2)
axes[1, 0].plot(history['val_iou'], label='Val', linewidth=2)
axes[1, 0].set_title('Train vs Val IoU', fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('IoU Score')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Learning Rate
axes[1, 1].plot(history['lrs'], c='red', linewidth=2)
axes[1, 1].set_title('Learning Rate Schedule', fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, "metrics", "training_history.png"), dpi=150)
plt.show()

# Save metrics to CSV
pd.DataFrame(history).to_csv(
    os.path.join(config.output_dir, "metrics", "training_history.csv"),
    index=False
)

print("✅ Training metrics saved")

## 14. Prediction & Visualization

In [ ]:
class SARPredictor:
    """Predictor for inference"""

    def __init__(self, checkpoint_path, config, device):
        self.device = device
        self.config = config

        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

        self.model = ViTSegmentation(
            config.img_size, config.patch_size, config.in_channels,
            config.out_channels, config.embed_dim, config.depth, config.num_heads
        ).to(device)

        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()

        self.transform = get_validation_augmentation()

    def predict(self, img_path, threshold=0.5):
        """Predict water mask"""
        with rasterio.open(img_path) as src:
            bands = list(range(1, min(self.config.in_channels + 1, src.count + 1)))
            image = src.read(bands).transpose(1, 2, 0).astype(np.float32)

            # Ensure correct number of channels
            if image.shape[2] < self.config.in_channels:
                pad = np.zeros((image.shape[0], image.shape[1],
                              self.config.in_channels - image.shape[2]), dtype=np.float32)
                image = np.concatenate([image, pad], axis=2)
            elif image.shape[2] > self.config.in_channels:
                image = image[:, :, :self.config.in_channels]

            if image.max() > 1.0:
                image /= 255.0

            orig_shape = (src.height, src.width)

        # Resize if needed
        if image.shape[0] != self.config.img_size or image.shape[1] != self.config.img_size:
            image = resize(image, (self.config.img_size, self.config.img_size, self.config.in_channels),
                         mode='reflect', anti_aliasing=True, preserve_range=True).astype(np.float32)

        # Transform and predict
        transformed = self.transform(image=image)
        image_tensor = transformed['image'].unsqueeze(0).to(self.device)

        with torch.no_grad():
            output = self.model(image_tensor)
            prob_map = torch.sigmoid(output).cpu().numpy()[0, 0]

        binary_mask = (prob_map > threshold).astype(np.uint8)

        return binary_mask, prob_map, orig_shape


# Load predictor
predictor = SARPredictor(best_checkpoint_path, config, DEVICE)
print("✅ Predictor loaded")

In [ ]:
# Visualize predictions on validation samples
def visualize_predictions(predictor, val_loader, num_samples=4):
    """Visualize model predictions"""
    model = predictor.model
    model.eval()

    images, masks = next(iter(val_loader))
    images = images.to(DEVICE)

    with torch.no_grad():
        outputs = torch.sigmoid(model(images)).cpu().numpy()

    images = images.cpu().numpy()
    masks = masks.cpu().numpy()

    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 4))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    for i in range(min(num_samples, len(images))):
        img = images[i].transpose(1, 2, 0)
        img = (img * 0.5 + 0.5).clip(0, 1)

        # VV channel
        axes[i, 0].imshow(img[:, :, 0], cmap='gray')
        axes[i, 0].set_title(f'Sample {i+1} - VV')
        axes[i, 0].axis('off')

        # VH channel
        axes[i, 1].imshow(img[:, :, 1], cmap='gray')
        axes[i, 1].set_title(f'Sample {i+1} - VH')
        axes[i, 1].axis('off')

        # Ground truth
        axes[i, 2].imshow(masks[i, 0], cmap='Blues')
        axes[i, 2].set_title('Ground Truth')
        axes[i, 2].axis('off')

        # Prediction
        axes[i, 3].imshow(outputs[i, 0], cmap='Blues')
        axes[i, 3].set_title('Prediction')
        axes[i, 3].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, "metrics", "predictions_visualization.png"), dpi=150)
    plt.show()

visualize_predictions(predictor, val_loader, num_samples=4)

## 15. Evaluation Metrics Summary

In [ ]:
# Calculate final metrics on validation set
print("\n📊 Computing final validation metrics...")

model.eval()
all_metrics = []

with torch.no_grad():
    for images, masks in tqdm(val_loader, desc="Evaluating"):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        outputs = model(images)

        batch_metrics = calculate_metrics(outputs, masks)
        all_metrics.append(batch_metrics)

# Average metrics
final_metrics = {}
for key in all_metrics[0].keys():
    final_metrics[key] = np.mean([m[key] for m in all_metrics])

# Display results
print("\n" + "="*50)
print("📈 Final Validation Metrics")
print("="*50)
for metric, value in final_metrics.items():
    print(f"{metric.upper():15s}: {value:.4f}")
print("="*50)

# Save to file
metrics_df = pd.DataFrame([final_metrics])
metrics_df.to_csv(
    os.path.join(config.output_dir, "metrics", "final_validation_metrics.csv"),
    index=False
)

print(f"\n✅ All outputs saved to: {config.output_dir}")

## 16. Summary & Next Steps

In [ ]:
print("""
═══════════════════════════════════════════════════════════════════════
✅ TRAINING PIPELINE COMPLETE
═══════════════════════════════════════════════════════════════════════

📁 Output Structure:
   {}/
   ├── checkpoints/
   │   └── best_model.pth           # Best model checkpoint
   ├── metrics/
   │   ├── training_history.png     # Training curves
   │   ├── training_history.csv     # Detailed metrics
   │   ├── predictions_visualization.png
   │   └── final_validation_metrics.csv
   └── predictions/
       └── (prediction outputs)

📊 Model Performance:
   • Best Val Loss:  {:.4f}
   • Best Val Dice:  {:.4f}
   • Final Val IoU:  {:.4f}

🎯 Next Steps:
   1. Review training curves for overfitting
   2. Test on new Sentinel-1 images
   3. Fine-tune hyperparameters if needed
   4. Deploy model for production use

═══════════════════════════════════════════════════════════════════════
""".format(config.output_dir, best_val_loss, best_val_dice, final_metrics['iou']))

## Load the model

In [ ]:
# checkpoint = torch.load('/content/drive/MyDrive/SAR_water_combined_outputs/checkpoints/best_model.pth')
# model.load_state_dict(checkpoint['model_state_dict'])
# config = checkpoint['config']  # Get training configuration

## Inference

In [ ]:
def run_inference(model, img_path, device, img_size=512, threshold=0.5):
    """
    Simple inference function
    """
    import rasterio
    from skimage.transform import resize
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

    # Load image
    with rasterio.open(img_path) as src:
        # Read first 2 bands (VV, VH)
        image = src.read([1, 2]).transpose(1, 2, 0).astype(np.float32)

        # Normalize
        if image.max() > 1.0:
            image = image / 255.0

        original_shape = (src.height, src.width)
        transform = src.transform

    # Resize to model input size
    if image.shape[0] != img_size or image.shape[1] != img_size:
        image = resize(image, (img_size, img_size, 2),
                      mode='reflect', anti_aliasing=True, preserve_range=True)

    # Convert to tensor
    transform_fn = A.Compose([ToTensorV2()])
    image_tensor = transform_fn(image=image)['image'].unsqueeze(0).to(device)

    # Predict
    model.eval()
    with torch.no_grad():
        output = model(image_tensor)
        prob_map = torch.sigmoid(output).cpu().numpy()[0, 0]

    # Resize back to original size
    if prob_map.shape != original_shape:
        prob_map = resize(prob_map, original_shape, mode='reflect', anti_aliasing=True)

    # Apply threshold
    binary_mask = (prob_map > threshold).astype(np.uint8)

    return binary_mask, prob_map

# Usage
model = ViTSegmentation(...)
model.load_state_dict(torch.load('best_model.pth')['model_state_dict'])
model.to(DEVICE)

binary_mask, prob_map = run_inference(model, "/content/drive/MyDrive/SAR_water_images_and_labels/images/Копия S1GRD_part_19_12_29_20250219_20250228_ea553e5c38a04f8c87c38b4fdcd223a2.tif", DEVICE)

### Batch Inference

In [ ]:
def batch_inference(model, input_dir, output_dir, device, pattern="*_img.tif"):
    """
    Run inference on all images in a directory
    """
    import glob

    os.makedirs(output_dir, exist_ok=True)

    # Find all image files
    img_files = glob.glob(os.path.join(input_dir, pattern))

    for img_path in tqdm(img_files, desc="Running inference"):
        try:
            # Get base name
            base_name = os.path.basename(img_path).replace('_img.tif', '')

            # Run inference
            binary_mask, prob_map = run_inference(model, img_path, device)

            # Save results
            with rasterio.open(img_path) as src:
                profile = src.profile
                profile.update(dtype=rasterio.float32, count=1, compress='lzw')

            # Save probability map
            prob_output = os.path.join(output_dir, f"{base_name}_probability.tif")
            with rasterio.open(prob_output, 'w', **profile) as dst:
                dst.write(prob_map.astype(np.float32), 1)

            # Save binary mask
            mask_output = os.path.join(output_dir, f"{base_name}_mask.tif")
            with rasterio.open(mask_output, 'w', **profile) as dst:
                dst.write(binary_mask.astype(np.float32), 1)

        except Exception as e:
            print(f"Error processing {img_path}: {e}")

    print(f"✅ Saved predictions to {output_dir}")

# Usage
batch_inference(model, "/content/drive/MyDrive/SAR_water_images_and_labels/images", "/content/drive/MyDrive/SAR_water_images_and_labels/batch_predictions_with_vision_transformer_model", DEVICE)